# Branch F — employment security and formality clustering

This notebook clusters the 81 respondents in the corrected strict final-Branch-F
cohort who have a valid employment block. The one routing-corrupted Branch-F
record remains in the cleaned dataset but is explicitly excluded here.


## Method and guardrails

- Three original employment responses are compared with equal-weight mixed
  Gower distance; no combined employment-security score is created.
- K-medoids and spectral clustering are compared for `k=2–4`.
- One hundred matched 80% subsamples per algorithm form a 200-partition
  consensus ensemble.
- Final profiles come from K-medoids on `1 - co-association`.
- Employment status, contract satisfaction, intention to change jobs, and the
  conditional insecurity-concern answer are descriptors only.


## Theory and distance equation

The ILO School-to-Work Transition Survey treats contract conditions and job
security as core structural aspects of employment quality. Sector formality
describes the respondent's structural labour-market position. Income–hours
adequacy and job satisfaction are deliberately left for a separate facet.

For respondents $i$ and $j$, equal-weight Gower distance is:

$$d(i,j)=\frac{1}{3}\sum_{h=1}^{3}\delta_{ijh}$$

For nominal answers, $\delta=0$ when responses match and 1 otherwise. For
explicitly ordered answers, $\delta$ is the absolute difference between their
normalized rank positions. `Do not know` for job security is kept as an
unordered uncertainty category: two unknown answers match; unknown versus a
known answer has distance 1. This is a respondent-to-respondent distance, not a
employment-security score.


In [1]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
VALUES_PATH = ROOT / "outputs/nysc_survey_cleaned_values.csv"
LABELS_PATH = ROOT / "outputs/nysc_survey_cleaned_labels.csv"
OUT = ROOT / "outputs/branch_f_employment_security_clusters"
OUT.mkdir(parents=True, exist_ok=True)

CLUSTER_FEATURES = [
    "contract_type",
    "job_security_12months",
    "sector_formality",
]
FEATURE_TYPES = {
    "contract_type": "nominal",
    "job_security_12months": "ordinal_with_unknown",
    "sector_formality": "ordinal",
}
DESCRIPTOR_VARIABLES = [
    "employment_status",
    "contract_satisfaction",
    "hopes_to_change_job",
    "insecurity_bothers",
]
DESCRIPTOR_ROLES = {
    "employment_status": "work_arrangement_descriptor",
    "contract_satisfaction": "subjective_contract_descriptor",
    "hopes_to_change_job": "transition_intention_descriptor",
    "insecurity_bothers": "conditional_security_descriptor",
}
ALL_SOURCE_VARIABLES = CLUSTER_FEATURES + DESCRIPTOR_VARIABLES

RANDOM_STATE = 20260827
MIN_PROFILE_SIZE = 15
CANDIDATE_SUBSAMPLES = 50
CONSENSUS_SUBSAMPLES = 100
SUBSAMPLE_FRACTION = 0.80
ENSEMBLE_BOOTSTRAPS = 100


## 1. Rebuild and audit the valid strict final-Branch-F employment cohort


In [2]:
values_all = pd.read_csv(VALUES_PATH)
labels_all = pd.read_csv(LABELS_PATH)

strict_mask = (
    values_all["branch"].eq("F")
    & values_all["branch_routing_source"].eq("translated_final_active_branch")
    & ~values_all["branch_integrity_issue"].astype(bool)
    & values_all["employment_status"].notna()
)

values = values_all.loc[strict_mask, [
    "respondent_id",
    "entry_branch",
    "branch",
    "branch_routing_source",
    "branch_path_disagreement",
    "branch_integrity_issue",
    "current_activity_status",
] + ALL_SOURCE_VARIABLES].copy()

labels = labels_all[[
    "respondent_id", "current_activity_status"
] + ALL_SOURCE_VARIABLES].copy()

cohort = values.merge(
    labels,
    on="respondent_id",
    suffixes=("__value", "__label"),
    validate="one_to_one",
)

validation = {
    "records": int(len(cohort)),
    "unique_ids": int(cohort["respondent_id"].nunique()),
    "wrong_branch": int(cohort["branch"].ne("F").sum()),
    "wrong_routing_source": int(
        cohort["branch_routing_source"].ne("translated_final_active_branch").sum()
    ),
    "path_disagreements_remaining": int(cohort["branch_path_disagreement"].sum()),
    "integrity_issues_remaining": int(cohort["branch_integrity_issue"].sum()),
    "invalid_activity_status": int(
        (~cohort["current_activity_status__value"].isin(
            ["working", "temp_absent", "finished_plan_work"]
        )).sum()
    ),
}

assert validation["records"] == 81
assert validation["unique_ids"] == 81
assert validation["wrong_branch"] == 0
assert validation["wrong_routing_source"] == 0
assert validation["path_disagreements_remaining"] == 0
assert validation["integrity_issues_remaining"] == 0
assert validation["invalid_activity_status"] == 0

display(pd.Series(validation, name="value").to_frame())


,value
records,81
unique_ids,81
wrong_branch,0
wrong_routing_source,0
path_disagreements_remaining,0
integrity_issues_remaining,0
invalid_activity_status,0


## 2. Screen variables and verify response coverage

Sector formality has an explicit questionnaire ordering. Contract type remains
nominal because written/oral form and duration do not create one unambiguous
rank order. Job-security uncertainty is preserved explicitly.


In [3]:
screening_rows = []
for variable in ALL_SOURCE_VARIABLES:
    label_col = f"{variable}__label"
    observed = cohort[label_col].notna()
    counts = cohort.loc[observed, label_col].value_counts(dropna=False)
    screening_rows.append({
        "variable": variable,
        "role": "cluster_input" if variable in CLUSTER_FEATURES else DESCRIPTOR_ROLES[variable],
        "observed": int(observed.sum()),
        "missing": int((~observed).sum()),
        "coverage_percent": 100 * float(observed.mean()),
        "unique_observed_responses": int(counts.size),
        "dominant_response": str(counts.index[0]) if len(counts) else None,
        "dominant_response_percent": 100 * float(counts.iloc[0] / counts.sum()) if len(counts) else np.nan,
    })

variable_screening = pd.DataFrame(screening_rows)

security_code = pd.to_numeric(cohort["job_security_12months__value"], errors="coerce")
formality_code = pd.to_numeric(cohort["sector_formality__value"], errors="coerce")
validation.update({
    "cluster_input_missing_values": int(
        cohort[[f"{v}__label" for v in CLUSTER_FEATURES]].isna().sum().sum()
    ),
    "invalid_security_codes": int((~security_code.between(1, 4)).sum()),
    "invalid_formality_codes": int((~formality_code.isin([1, 2, 3, 5])).sum()),
})
assert validation["cluster_input_missing_values"] == 0
assert validation["invalid_security_codes"] == 0
assert validation["invalid_formality_codes"] == 0

display(variable_screening.round(2))
display(pd.Series(validation, name="value").to_frame())


,variable,role,observed,missing,coverage_percent,unique_observed_responses,dominant_response,dominant_response_percent
0,contract_type,cluster_input,81,0,100.00,8,Written contract — permanent/unlimited,49.38
1,job_security_12months,cluster_input,81,0,100.00,4,Very likely — my job is secure,45.68
2,sector_formality,cluster_input,81,0,100.00,4,"Fully formal — registered employer, written co...",62.96
3,employment_status,work_arrangement_descriptor,81,0,100.00,6,For an employer — for a salary or wage,76.54
4,contract_satisfaction,subjective_contract_descriptor,81,0,100.00,8,Satisfied — it gives me the flexibility I need,34.57
5,hopes_to_change_job,transition_intention_descriptor,81,0,100.00,3,Yes — I want to change,56.79
6,insecurity_bothers,conditional_security_descriptor,34,47,41.98,2,No — I accept it or it does not concern me,52.94


,value
records,81
unique_ids,81
wrong_branch,0
wrong_routing_source,0
path_disagreements_remaining,0
integrity_issues_remaining,0
invalid_activity_status,0
cluster_input_missing_values,0
invalid_security_codes,0
invalid_formality_codes,0


## 3. Build original-response features and Gower distance


In [4]:
# Ordinal transformations below are normalized questionnaire-rank positions used
# only inside Gower distance. They are not respondent-level job-quality scores.
features = pd.DataFrame(index=cohort.index)
features["contract_type"] = cohort["contract_type__label"].astype("string")
features["job_security_12months"] = security_code.map({1: 1.0, 2: 0.5, 3: 0.0, 4: -1.0})
features["sector_formality"] = formality_code.map({1: 1.0, 2: 2/3, 3: 1/3, 5: 0.0})

feature_audit = pd.DataFrame({
    "feature": CLUSTER_FEATURES,
    "type": [FEATURE_TYPES[c] for c in CLUSTER_FEATURES],
    "observed": [int(features[c].notna().sum()) for c in CLUSTER_FEATURES],
    "missing": [int(features[c].isna().sum()) for c in CLUSTER_FEATURES],
    "maximum_distance_weight": [1 / len(CLUSTER_FEATURES)] * len(CLUSTER_FEATURES),
})
assert features.notna().sum(axis=1).min() == len(CLUSTER_FEATURES)
display(feature_audit)


def mixed_gower(frame, columns=None):
    if columns is None:
        columns = list(frame.columns)
    n = len(frame)
    numerator = np.zeros((n, n), dtype=float)
    denominator = np.zeros((n, n), dtype=float)
    for column in columns:
        feature_type = FEATURE_TYPES[column]
        if feature_type == "nominal":
            series = frame[column].astype("string")
            observed = series.notna().to_numpy()
            values_k = series.fillna("__MISSING__").to_numpy(dtype=str)
            difference = (values_k[:, None] != values_k[None, :]).astype(float)
        elif feature_type == "ordinal_with_unknown":
            series = pd.to_numeric(frame[column], errors="coerce")
            observed = series.notna().to_numpy()
            values_k = series.fillna(-1).to_numpy(dtype=float)
            unknown = values_k < 0
            both_unknown = unknown[:, None] & unknown[None, :]
            one_unknown = unknown[:, None] ^ unknown[None, :]
            difference = np.abs(values_k[:, None] - values_k[None, :])
            difference = np.where(both_unknown, 0.0, difference)
            difference = np.where(one_unknown, 1.0, difference)
        else:
            series = pd.to_numeric(frame[column], errors="coerce")
            observed = series.notna().to_numpy()
            values_k = series.fillna(0).to_numpy(dtype=float)
            difference = np.abs(values_k[:, None] - values_k[None, :])
        pair_observed = observed[:, None] & observed[None, :]
        numerator += np.where(pair_observed, difference, 0.0)
        denominator += pair_observed
    distance = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0,
    )
    np.fill_diagonal(distance, 0.0)
    return distance, denominator


D, pair_feature_coverage = mixed_gower(features)
assert np.allclose(D, D.T)
assert D.min() >= 0 and D.max() <= 1
assert pair_feature_coverage.min() == len(CLUSTER_FEATURES)

positive_distances = D[D > 0]
sigma = float(np.median(positive_distances))


def affinity(distance):
    result = np.exp(-(distance ** 2) / (2 * sigma ** 2))
    np.fill_diagonal(result, 1.0)
    return result


print("Gower distance median:", round(float(np.median(positive_distances)), 4))
print(
    "Pairwise observed features: min",
    int(pair_feature_coverage.min()),
    "median",
    int(np.median(pair_feature_coverage)),
    "max",
    int(pair_feature_coverage.max()),
)


,feature,type,observed,missing,maximum_distance_weight
0,contract_type,nominal,81,0,0.333333
1,job_security_12months,ordinal_with_unknown,81,0,0.333333
2,sector_formality,ordinal,81,0,0.333333


Gower distance median: 0.5556
Pairwise observed features: min 3 median 3 max 3


## 4. Compare candidate profile counts


In [5]:
def fit_kmedoids(distance, k, n_init=200, max_iter=200, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = len(distance)
    best = None
    for _ in range(n_init):
        medoids = np.sort(rng.choice(n, size=k, replace=False))
        for _ in range(max_iter):
            labels_k = np.argmin(distance[:, medoids], axis=1)
            updated = medoids.copy()
            for cluster in range(k):
                members = np.flatnonzero(labels_k == cluster)
                if len(members) == 0:
                    nearest = distance[:, medoids].min(axis=1)
                    available = np.setdiff1d(np.argsort(nearest)[::-1], updated)
                    updated[cluster] = available[0]
                else:
                    within = distance[np.ix_(members, members)]
                    updated[cluster] = members[np.argmin(within.sum(axis=1))]
            if np.array_equal(updated, medoids):
                break
            medoids = updated
        labels_k = np.argmin(distance[:, medoids], axis=1)
        loss = float(distance[np.arange(n), medoids[labels_k]].sum())
        candidate = (loss, tuple(medoids.tolist()), medoids.copy(), labels_k.copy())
        if best is None or candidate[:2] < best[:2]:
            best = candidate
    return {"labels": best[3], "medoids": best[2], "loss": best[0]}


def medoids_from_labels(distance, labels_k):
    result = []
    for cluster in sorted(np.unique(labels_k)):
        members = np.flatnonzero(labels_k == cluster)
        within = distance[np.ix_(members, members)]
        result.append(members[np.argmin(within.sum(axis=1))])
    return np.asarray(result, dtype=int)


def subsample_stability(algorithm, k, reference_labels):
    offset = 0 if algorithm == "kmedoids" else 10000
    rng = np.random.default_rng(RANDOM_STATE + 100 * k + offset)
    rows = []
    sample_size = int(np.floor(SUBSAMPLE_FRACTION * len(D)))
    for repetition in range(CANDIDATE_SUBSAMPLES):
        idx = np.sort(rng.choice(len(D), size=sample_size, replace=False))
        D_sub = D[np.ix_(idx, idx)]
        if algorithm == "kmedoids":
            result = fit_kmedoids(
                D_sub, k, n_init=50,
                random_state=RANDOM_STATE + 1000 * k + repetition,
            )
            medoids = idx[result["medoids"]]
        else:
            labels_sub = SpectralClustering(
                n_clusters=k,
                affinity="precomputed",
                assign_labels="kmeans",
                n_init=30,
                random_state=RANDOM_STATE + 2000 * k + repetition,
            ).fit_predict(affinity(D_sub))
            medoids = idx[medoids_from_labels(D_sub, labels_sub)]
        predicted = np.argmin(D[:, medoids], axis=1)
        rows.append(adjusted_rand_score(reference_labels, predicted))
    return np.asarray(rows)


candidate_rows = []
candidate_stability_rows = []
full_source_solutions = {}

for k in range(2, 5):
    kmedoids = fit_kmedoids(D, k, n_init=400, random_state=RANDOM_STATE + k)
    spectral_labels = SpectralClustering(
        n_clusters=k,
        affinity="precomputed",
        assign_labels="kmeans",
        n_init=100,
        random_state=RANDOM_STATE + 100 + k,
    ).fit_predict(affinity(D))
    full_source_solutions[("kmedoids", k)] = kmedoids["labels"]
    full_source_solutions[("spectral", k)] = spectral_labels
    method_values = {}

    for algorithm, labels_k in [
        ("kmedoids", kmedoids["labels"]),
        ("spectral", spectral_labels),
    ]:
        stability = subsample_stability(algorithm, k, labels_k)
        method_values[algorithm] = {
            "silhouette": silhouette_score(D, labels_k, metric="precomputed"),
            "stability": float(stability.mean()),
            "stability_p10": float(np.quantile(stability, 0.10)),
            "minimum_size": int(pd.Series(labels_k).value_counts().min()),
        }
        candidate_stability_rows.extend({
            "algorithm": algorithm,
            "k": k,
            "repetition": repetition + 1,
            "ari": value,
        } for repetition, value in enumerate(stability))

    candidate_rows.append({
        "k": k,
        "kmedoids_silhouette": method_values["kmedoids"]["silhouette"],
        "spectral_silhouette": method_values["spectral"]["silhouette"],
        "mean_source_silhouette": np.mean([
            method_values["kmedoids"]["silhouette"],
            method_values["spectral"]["silhouette"],
        ]),
        "kmedoids_stability_mean": method_values["kmedoids"]["stability"],
        "spectral_stability_mean": method_values["spectral"]["stability"],
        "mean_source_stability": np.mean([
            method_values["kmedoids"]["stability"],
            method_values["spectral"]["stability"],
        ]),
        "minimum_source_p10_stability": min(
            method_values["kmedoids"]["stability_p10"],
            method_values["spectral"]["stability_p10"],
        ),
        "source_algorithm_ari": adjusted_rand_score(kmedoids["labels"], spectral_labels),
        "minimum_cluster_size_across_sources": min(
            method_values["kmedoids"]["minimum_size"],
            method_values["spectral"]["minimum_size"],
        ),
    })

candidate_metrics = pd.DataFrame(candidate_rows)
candidate_stability = pd.DataFrame(candidate_stability_rows)
eligible = candidate_metrics.loc[
    candidate_metrics["minimum_cluster_size_across_sources"].ge(MIN_PROFILE_SIZE)
].copy()
if eligible.empty:
    raise RuntimeError("No k=2–5 candidate meets the minimum profile-size rule.")

eligible["rank_mean_silhouette"] = eligible["mean_source_silhouette"].rank(
    ascending=False, method="min"
)
eligible["rank_mean_stability"] = eligible["mean_source_stability"].rank(
    ascending=False, method="min"
)
eligible["primary_mean_rank"] = eligible[[
    "rank_mean_silhouette", "rank_mean_stability"
]].mean(axis=1)
best_rank = eligible["primary_mean_rank"].min()
selected_k = int(
    eligible.loc[eligible["primary_mean_rank"].eq(best_rank)]
    .sort_values("k").iloc[0]["k"]
)
candidate_metrics = candidate_metrics.merge(
    eligible[["k", "primary_mean_rank"]], on="k", how="left"
)
candidate_metrics["selected"] = candidate_metrics["k"].eq(selected_k)

display(candidate_metrics.round(4))
print("Selected common k:", selected_k)


,k,kmedoids_silhouette,spectral_silhouette,mean_source_silhouette,kmedoids_stability_mean,spectral_stability_mean,mean_source_stability,minimum_source_p10_stability,source_algorithm_ari,minimum_cluster_size_across_sources,primary_mean_rank,selected
0,2,0.3384,0.3768,0.3576,0.8347,0.7078,0.7713,0.3754,0.8086,26,1.5,True
1,3,0.4140,0.4283,0.4211,0.7804,0.6991,0.7397,0.4796,0.6444,15,1.5,False
2,4,0.4757,0.4106,0.4431,0.6591,0.5009,0.5800,0.3252,0.6011,9,NaN,False


Selected common k: 2


## 5. Build the K-medoids + spectral consensus ensemble


In [6]:
ensemble_rng = np.random.default_rng(RANDOM_STATE + 30000)
n = len(D)
sample_size = int(np.floor(SUBSAMPLE_FRACTION * n))
cover_matrices = []
same_matrices = []
base_rows = []

for repetition in range(CONSENSUS_SUBSAMPLES):
    idx = np.sort(ensemble_rng.choice(n, size=sample_size, replace=False))
    D_sub = D[np.ix_(idx, idx)]
    cover = np.zeros((n, n), dtype=np.uint8)
    cover[np.ix_(idx, idx)] = 1

    kmedoids_sub = fit_kmedoids(
        D_sub, selected_k, n_init=100,
        random_state=RANDOM_STATE + 40000 + repetition,
    )["labels"]
    spectral_sub = SpectralClustering(
        n_clusters=selected_k,
        affinity="precomputed",
        assign_labels="kmeans",
        n_init=30,
        random_state=RANDOM_STATE + 50000 + repetition,
    ).fit_predict(affinity(D_sub))

    for algorithm, labels_sub in [
        ("kmedoids", kmedoids_sub),
        ("spectral", spectral_sub),
    ]:
        same = np.zeros((n, n), dtype=np.uint8)
        same[np.ix_(idx, idx)] = (
            labels_sub[:, None] == labels_sub[None, :]
        ).astype(np.uint8)
        cover_matrices.append(cover.copy())
        same_matrices.append(same)
        base_rows.append({
            "partition_id": len(base_rows) + 1,
            "algorithm": algorithm,
            "repetition": repetition + 1,
            "silhouette_in_subsample": silhouette_score(
                D_sub, labels_sub, metric="precomputed"
            ),
            "minimum_cluster_size": int(pd.Series(labels_sub).value_counts().min()),
        })

cover_stack = np.stack(cover_matrices)
same_stack = np.stack(same_matrices)
coverage = cover_stack.sum(axis=0)
together = same_stack.sum(axis=0)
consensus = together / coverage
np.fill_diagonal(consensus, 1.0)
consensus_distance = 1 - consensus
np.fill_diagonal(consensus_distance, 0.0)
base_partition_diagnostics = pd.DataFrame(base_rows)

final_fit = fit_kmedoids(
    consensus_distance, selected_k, n_init=500,
    random_state=RANDOM_STATE + 60000,
)
raw_labels = final_fit["labels"]

raw_medoids = {}
for raw_cluster in sorted(np.unique(raw_labels)):
    members = np.flatnonzero(raw_labels == raw_cluster)
    within = consensus_distance[np.ix_(members, members)]
    raw_medoids[int(raw_cluster)] = members[np.argmin(within.sum(axis=1))]

raw_order = sorted(
    raw_medoids,
    key=lambda label: str(cohort.iloc[raw_medoids[label]]["respondent_id"]),
)
raw_to_profile = {raw: profile + 1 for profile, raw in enumerate(raw_order)}
profile_id = np.asarray([raw_to_profile[int(label)] for label in raw_labels])
profile_names = {
    1: "Temporary/no-contract employment with uncertain security",
    2: "Predominantly permanent employment with stronger security",
}
assert set(np.unique(profile_id)) == set(profile_names), "Update descriptive profile names if selected k changes."

print("Base partitions:", len(base_partition_diagnostics))
print(
    "Pair coverage: min", int(coverage.min()),
    "median", int(np.median(coverage)),
    "max", int(coverage.max()),
)


Base partitions: 200
Pair coverage: min 90 median 124 max 178


## 6. Profile the final employment security clusters


In [7]:
assignments = cohort[[
    "respondent_id", "entry_branch", "branch", "branch_routing_source"
]].copy()
for variable in ALL_SOURCE_VARIABLES:
    assignments[f"{variable}__value"] = cohort[f"{variable}__value"]
    assignments[f"{variable}__response"] = cohort[f"{variable}__label"]

assignments["employment_security_profile_id"] = profile_id
assignments["employment_security_profile_label"] = assignments["employment_security_profile_id"].map(profile_names)
gower_individual_silhouette = silhouette_samples(D, profile_id, metric="precomputed")
consensus_individual_silhouette = silhouette_samples(
    consensus_distance, profile_id, metric="precomputed"
)
assignments["gower_individual_silhouette"] = gower_individual_silhouette
assignments["consensus_individual_silhouette"] = consensus_individual_silhouette

profile_sizes = (
    assignments.groupby(["employment_security_profile_id", "employment_security_profile_label"], as_index=False)
    .agg(
        records=("respondent_id", "size"),
        low_gower_confidence_records=(
            "gower_individual_silhouette", lambda values: int((values < 0.10).sum())
        ),
        mean_gower_silhouette=("gower_individual_silhouette", "mean"),
        mean_consensus_silhouette=("consensus_individual_silhouette", "mean"),
    )
)
profile_sizes["percent"] = 100 * profile_sizes["records"] / len(assignments)

response_rows = []
for variable in ALL_SOURCE_VARIABLES:
    response_col = f"{variable}__response"
    temp = (
        assignments.groupby(["employment_security_profile_id", response_col], dropna=False)
        .size().rename("records").reset_index()
    )
    temp["within_profile_percent"] = (
        100 * temp["records"]
        / temp.groupby("employment_security_profile_id")["records"].transform("sum")
    )
    temp = temp.rename(columns={response_col: "response"})
    temp.insert(1, "variable", variable)
    temp.insert(
        2,
        "role",
        "cluster_input" if variable in CLUSTER_FEATURES else DESCRIPTOR_ROLES[variable],
    )
    response_rows.append(temp)

profile_response_distributions = pd.concat(response_rows, ignore_index=True)
profile_modes = (
    profile_response_distributions.sort_values(
        ["employment_security_profile_id", "variable", "within_profile_percent"],
        ascending=[True, True, False],
    ).groupby(["employment_security_profile_id", "variable"], as_index=False).first()
)

medoid_rows = []
for raw_cluster, medoid_idx in raw_medoids.items():
    pid = raw_to_profile[raw_cluster]
    row = {
        "employment_security_profile_id": pid,
        "respondent_id": cohort.iloc[medoid_idx]["respondent_id"],
    }
    for feature in CLUSTER_FEATURES:
        row[f"{feature}__response"] = cohort.iloc[medoid_idx][f"{feature}__label"]
    medoid_rows.append(row)
medoid_profiles = pd.DataFrame(medoid_rows).sort_values("employment_security_profile_id")

contract_formality_crosstab = pd.crosstab(
    cohort["contract_type__label"],
    cohort["sector_formality__label"],
    dropna=False,
)

display(profile_sizes.round(3))
display(profile_modes.round(2))
display(medoid_profiles)


,employment_security_profile_id,employment_security_profile_label,records,low_gower_confidence_records,mean_gower_silhouette,mean_consensus_silhouette,percent
0,1,Temporary/no-contract employment with uncertai...,29,9,0.153,0.690,35.802
1,2,Predominantly permanent employment with strong...,52,1,0.487,0.934,64.198


,employment_security_profile_id,variable,role,response,records,within_profile_percent
0,1,contract_satisfaction,subjective_contract_descriptor,Satisfied — it gives me the flexibility I need,10,34.48
1,1,contract_type,cluster_input,Written contract — temporary (under 12 months),12,41.38
2,1,employment_status,work_arrangement_descriptor,For an employer — for a salary or wage,17,58.62
3,1,hopes_to_change_job,transition_intention_descriptor,Yes — I want to change,18,62.07
4,1,insecurity_bothers,conditional_security_descriptor,No — I accept it or it does not concern me,11,37.93
5,1,job_security_12months,cluster_input,Likely but not certain — I have some concern,15,51.72
6,1,sector_formality,cluster_input,"Fully formal — registered employer, written co...",16,55.17
7,2,contract_satisfaction,subjective_contract_descriptor,Satisfied — it gives me the job security I need,19,36.54
8,2,contract_type,cluster_input,Written contract — permanent/unlimited,40,76.92
9,2,employment_status,work_arrangement_descriptor,For an employer — for a salary or wage,45,86.54


,employment_security_profile_id,respondent_id,contract_type__response,job_security_12months__response,sector_formality__response
1,1,nysc-33aaf2f7-dc2a-4530-ae73-3bfdf5be63ab,Oral contract — temporary (12–36 months),Not likely — I doubt I will still have this job,Mostly formal — some documentation but not ful...
0,2,nysc-c74b80fb-a56d-43d8-85a7-8b1965db1fba,Written contract — permanent/unlimited,Likely but not certain — I have some concern,Mostly formal — some documentation but not ful...


## 7. Validate stability, ambiguity, and feature sensitivity


In [8]:
confidence_rows = []
for i in range(n):
    own = profile_id[i]
    own_members = np.flatnonzero(profile_id == own)
    own_members = own_members[own_members != i]
    within = float(consensus[i, own_members].mean()) if len(own_members) else 1.0
    strongest_other = max(
        float(consensus[i, np.flatnonzero(profile_id == other)].mean())
        for other in sorted(set(profile_id) - {own})
    )
    confidence_rows.append({
        "respondent_id": cohort.iloc[i]["respondent_id"],
        "employment_security_profile_id": int(own),
        "within_profile_consensus": within,
        "strongest_other_profile_consensus": strongest_other,
        "consensus_margin": within - strongest_other,
        "ambiguous_consensus_membership": (
            (within < 0.70) or ((within - strongest_other) < 0.20)
        ),
    })

respondent_confidence = pd.DataFrame(confidence_rows)
assignments = assignments.merge(
    respondent_confidence.drop(columns="employment_security_profile_id"),
    on="respondent_id", validate="one_to_one",
)

bootstrap_rng = np.random.default_rng(RANDOM_STATE + 70000)
ensemble_stability_rows = []
for repetition in range(ENSEMBLE_BOOTSTRAPS):
    selected = bootstrap_rng.choice(len(cover_stack), size=len(cover_stack), replace=True)
    boot_cover = cover_stack[selected].sum(axis=0)
    boot_same = same_stack[selected].sum(axis=0)
    boot_consensus = np.divide(
        boot_same, boot_cover,
        out=np.zeros_like(boot_same, dtype=float),
        where=boot_cover > 0,
    )
    np.fill_diagonal(boot_consensus, 1.0)
    boot_distance = 1 - boot_consensus
    np.fill_diagonal(boot_distance, 0.0)
    boot_labels = fit_kmedoids(
        boot_distance, selected_k, n_init=80,
        random_state=RANDOM_STATE + 80000 + repetition,
    )["labels"]
    ensemble_stability_rows.append({
        "repetition": repetition + 1,
        "ari_vs_full_consensus": adjusted_rand_score(profile_id, boot_labels),
    })

ensemble_stability = pd.DataFrame(ensemble_stability_rows)
full_kmedoids = full_source_solutions[("kmedoids", selected_k)]
full_spectral = full_source_solutions[("spectral", selected_k)]
consensus_hac_average = AgglomerativeClustering(
    n_clusters=selected_k, metric="precomputed", linkage="average"
).fit_predict(consensus_distance)
consensus_hac_complete = AgglomerativeClustering(
    n_clusters=selected_k, metric="precomputed", linkage="complete"
).fit_predict(consensus_distance)

upper = consensus[np.triu_indices(n, 1)]
metrics = {
    "gower_silhouette": float(silhouette_score(D, profile_id, metric="precomputed")),
    "consensus_distance_silhouette": float(
        silhouette_score(consensus_distance, profile_id, metric="precomputed")
    ),
    "ensemble_bootstrap_ari_mean": float(
        ensemble_stability["ari_vs_full_consensus"].mean()
    ),
    "ensemble_bootstrap_ari_p10": float(
        ensemble_stability["ari_vs_full_consensus"].quantile(0.10)
    ),
    "ari_vs_full_kmedoids": float(adjusted_rand_score(profile_id, full_kmedoids)),
    "ari_vs_full_spectral": float(adjusted_rand_score(profile_id, full_spectral)),
    "ari_consensus_kmedoids_vs_hac_average": float(
        adjusted_rand_score(profile_id, consensus_hac_average)
    ),
    "ari_consensus_kmedoids_vs_hac_complete": float(
        adjusted_rand_score(profile_id, consensus_hac_complete)
    ),
    "minimum_profile_size": int(pd.Series(profile_id).value_counts().min()),
    "low_gower_confidence_percent": 100 * float(
        np.mean(gower_individual_silhouette < 0.10)
    ),
    "ambiguous_consensus_percent": 100 * float(
        assignments["ambiguous_consensus_membership"].mean()
    ),
    "pac_10_90": float(np.mean((upper > 0.10) & (upper < 0.90))),
}

feature_sensitivity_rows = []
for omitted in CLUSTER_FEATURES:
    retained = [feature for feature in CLUSTER_FEATURES if feature != omitted]
    D_reduced, coverage_reduced = mixed_gower(features, retained)
    reduced_fit = fit_kmedoids(
        D_reduced, selected_k, n_init=400,
        random_state=RANDOM_STATE + 90000 + CLUSTER_FEATURES.index(omitted),
    )
    feature_sensitivity_rows.append({
        "omitted_feature": omitted,
        "retained_features": len(retained),
        "minimum_pair_coverage": int(coverage_reduced.min()),
        "gower_silhouette_reduced_kmedoids": float(
            silhouette_score(D_reduced, reduced_fit["labels"], metric="precomputed")
        ),
        "ari_reduced_kmedoids_vs_final_consensus": float(
            adjusted_rand_score(profile_id, reduced_fit["labels"])
        ),
    })
feature_sensitivity = pd.DataFrame(feature_sensitivity_rows)

quality_checks = pd.DataFrame([
    {"check": "original Gower silhouette >= 0.25", "value": metrics["gower_silhouette"], "passed": metrics["gower_silhouette"] >= 0.25},
    {"check": "ensemble bootstrap mean ARI >= 0.80", "value": metrics["ensemble_bootstrap_ari_mean"], "passed": metrics["ensemble_bootstrap_ari_mean"] >= 0.80},
    {"check": f"minimum profile size >= {MIN_PROFILE_SIZE}", "value": metrics["minimum_profile_size"], "passed": metrics["minimum_profile_size"] >= MIN_PROFILE_SIZE},
    {"check": "ARI with full K-medoids >= 0.60", "value": metrics["ari_vs_full_kmedoids"], "passed": metrics["ari_vs_full_kmedoids"] >= 0.60},
    {"check": "ARI with full spectral >= 0.60", "value": metrics["ari_vs_full_spectral"], "passed": metrics["ari_vs_full_spectral"] >= 0.60},
    {"check": "PAC (0.10, 0.90) <= 0.20", "value": metrics["pac_10_90"], "passed": metrics["pac_10_90"] <= 0.20},
    {"check": "ambiguous consensus respondents <= 20%", "value": metrics["ambiguous_consensus_percent"], "passed": metrics["ambiguous_consensus_percent"] <= 20},
    {"check": "low-confidence respondents in original Gower space <= 20%", "value": metrics["low_gower_confidence_percent"], "passed": metrics["low_gower_confidence_percent"] <= 20},
])

primary_pass = bool(quality_checks.iloc[:5]["passed"].all())
if primary_pass and quality_checks.iloc[5:]["passed"].all():
    statistical_status = "Stable, cross-method-supported employment-security segmentation"
elif primary_pass:
    statistical_status = "Passes primary stability rules but retains ambiguity cautions"
else:
    statistical_status = "No sufficiently robust employment-security segmentation found"

status = statistical_status

display(pd.Series(metrics, name="value").to_frame().round(3))
display(quality_checks.round(3))
display(feature_sensitivity.round(3))
print("Statistical result:", statistical_status)


,value
gower_silhouette,0.368
consensus_distance_silhouette,0.846
ensemble_bootstrap_ari_mean,0.953
ensemble_bootstrap_ari_p10,0.902
ari_vs_full_kmedoids,0.854
ari_vs_full_spectral,0.950
ari_consensus_kmedoids_vs_hac_average,0.950
ari_consensus_kmedoids_vs_hac_complete,0.854
minimum_profile_size,29.000
low_gower_confidence_percent,12.346


,check,value,passed
0,original Gower silhouette >= 0.25,0.368,True
1,ensemble bootstrap mean ARI >= 0.80,0.953,True
2,minimum profile size >= 15,29.000,True
3,ARI with full K-medoids >= 0.60,0.854,True
4,ARI with full spectral >= 0.60,0.950,True
5,"PAC (0.10, 0.90) <= 0.20",0.427,False
6,ambiguous consensus respondents <= 20%,6.173,True
7,low-confidence respondents in original Gower s...,12.346,True


,omitted_feature,retained_features,minimum_pair_coverage,gower_silhouette_reduced_kmedoids,ari_reduced_kmedoids_vs_final_consensus
0,contract_type,2,2,0.464,0.329
1,job_security_12months,2,2,0.450,0.193
2,sector_formality,2,2,0.396,0.854


Statistical result: Passes primary stability rules but retains ambiguity cautions


## 8. Visualise candidate evidence, consensus, and profiles


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(candidate_metrics["k"], candidate_metrics["mean_source_silhouette"], marker="o", label="mean source silhouette")
axes[0].axvline(selected_k, color="tab:red", linestyle="--", label=f"selected k={selected_k}")
axes[0].axhline(0.25, color="grey", linestyle=":")
axes[0].set(title="Candidate separation", xlabel="Number of profiles", ylabel="Mean Gower silhouette")
axes[0].legend()
axes[1].plot(candidate_metrics["k"], candidate_metrics["mean_source_stability"], marker="o", label="mean source stability")
axes[1].plot(candidate_metrics["k"], candidate_metrics["source_algorithm_ari"], marker="s", label="K-medoids vs spectral ARI")
axes[1].axvline(selected_k, color="tab:red", linestyle="--")
axes[1].axhline(0.70, color="grey", linestyle=":")
axes[1].set(title="Stability and source agreement", xlabel="Number of profiles", ylabel="ARI", ylim=(-0.05, 1.05))
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "employment_security_candidate_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()

order = np.argsort(profile_id, kind="stable")
fig, ax = plt.subplots(figsize=(8, 7))
heat = ax.imshow(consensus[np.ix_(order, order)], cmap="viridis", vmin=0, vmax=1, aspect="auto")
boundaries = np.cumsum(pd.Series(profile_id[order]).value_counts(sort=False).sort_index().to_numpy())[:-1]
for boundary in boundaries:
    ax.axhline(boundary - 0.5, color="white", linewidth=1.2)
    ax.axvline(boundary - 0.5, color="white", linewidth=1.2)
ax.set(title="Employment-security/formality consensus matrix", xlabel="Respondents ordered by profile", ylabel="Respondents")
fig.colorbar(heat, ax=ax, label="Proportion clustered together")
fig.tight_layout()
fig.savefig(OUT / "employment_security_consensus_matrix_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()

J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (D ** 2) @ J
eigenvalues, eigenvectors = np.linalg.eigh(B)
eigen_order = np.argsort(eigenvalues)[::-1]
positive = eigen_order[eigenvalues[eigen_order] > 0][:2]
coordinates = eigenvectors[:, positive] * np.sqrt(eigenvalues[positive])

fig, ax = plt.subplots(figsize=(8, 6))
plot_points = pd.DataFrame({
    "x": np.round(coordinates[:, 0], 10),
    "y": np.round(coordinates[:, 1], 10),
    "employment_security_profile_id": profile_id,
})
plot_bubbles = (
    plot_points.groupby(["employment_security_profile_id", "x", "y"], as_index=False)
    .size().rename(columns={"size": "respondents"})
)
for pid in sorted(np.unique(profile_id)):
    group = plot_bubbles.loc[plot_bubbles["employment_security_profile_id"].eq(pid)]
    ax.scatter(
        group["x"], group["y"],
        s=45 + 10 * group["respondents"],
        alpha=0.72,
        edgecolor="white",
        linewidth=0.8,
        label=profile_names[int(pid)],
    )
    for row in group.itertuples():
        ax.annotate(
            f"n={row.respondents}", (row.x, row.y),
            ha="center", va="center", fontsize=7,
        )
ax.set(
    title="Employment security response patterns in two-dimensional Gower space",
    xlabel="Classical MDS axis 1 (display coordinate only)",
    ylabel="Classical MDS axis 2 (display coordinate only)",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "employment_security_profile_scatterplot.png", dpi=180, bbox_inches="tight")
plt.show()


/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_3253/541482399.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_3253/541482399.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_3253/541482399.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Save reproducible outputs


In [10]:
variable_screening.to_csv(OUT / "employment_security_variable_screening.csv", index=False)
feature_audit.to_csv(OUT / "employment_security_feature_audit.csv", index=False)
candidate_metrics.to_csv(OUT / "employment_security_candidate_metrics.csv", index=False)
candidate_stability.to_csv(OUT / "employment_security_candidate_subsample_stability.csv", index=False)
base_partition_diagnostics.to_csv(OUT / "employment_security_consensus_base_partitions.csv", index=False)
assignments.to_csv(OUT / "employment_security_consensus_assignments.csv", index=False)
profile_sizes.to_csv(OUT / "employment_security_profile_sizes.csv", index=False)
profile_response_distributions.to_csv(OUT / "employment_security_profile_response_distributions.csv", index=False)
profile_modes.to_csv(OUT / "employment_security_profile_modes.csv", index=False)
medoid_profiles.to_csv(OUT / "employment_security_medoid_profiles.csv", index=False)
contract_formality_crosstab.to_csv(OUT / "employment_security_contract_by_formality.csv")
respondent_confidence.to_csv(OUT / "employment_security_respondent_confidence.csv", index=False)
ensemble_stability.to_csv(OUT / "employment_security_consensus_ensemble_stability.csv", index=False)
feature_sensitivity.to_csv(OUT / "employment_security_leave_one_feature_out.csv", index=False)
quality_checks.to_csv(OUT / "employment_security_quality_checks.csv", index=False)

consensus_export = pd.DataFrame(consensus, columns=cohort["respondent_id"])
consensus_export.insert(0, "respondent_id", cohort["respondent_id"].to_numpy())
consensus_export.to_csv(OUT / "employment_security_consensus_matrix.csv", index=False)

summary = {
    "cohort": {
        "records": int(len(cohort)),
        "branch": "F",
        "routing_source": "translated_final_active_branch",
        "strict_final_branch_records": 82,
        "excluded_routing_integrity_records": 1,
    },
    "dimension": "employment security and formality",
    "clustering_features": CLUSTER_FEATURES,
    "descriptors": DESCRIPTOR_VARIABLES,
    "data_quality": validation,
    "method": {
        "distance": "equal-weight mixed Gower on three original employment responses",
        "candidate_k": [2, 3, 4],
        "selected_k": selected_k,
        "ensemble": "equal-weight K-medoids plus spectral consensus",
        "base_partitions": 2 * CONSENSUS_SUBSAMPLES,
        "final_method": "K-medoids on one minus co-association",
    },
    "profiles": {
        "sizes": {
            str(int(row.employment_security_profile_id)): int(row.records)
            for row in profile_sizes.itertuples()
        },
        "names": {str(key): value for key, value in profile_names.items()},
    },
    "metrics": metrics,
    "decision": {
        "primary_checks_passed": primary_pass,
        "checks_passed": int(quality_checks["passed"].sum()),
        "checks_total": int(len(quality_checks)),
        "included_in_final_dimension_set": primary_pass,
        "status": status,
    },
    "limitations": [
        "These are exploratory employment-security/formality patterns, not a validated job-quality scale.",
        "Ordinal Gower positions preserve questionnaire order only and are not respondent-level interval scores.",
        "Contract type remains nominal because written/oral form and duration do not create one unambiguous rank order.",
        "Job-security 'do not know' is treated as an uncertainty category, not forced above or below a known response.",
        "PAC remains above the conservative 0.20 threshold, so some respondent pairs have uncertain co-membership even though the primary separation and stability checks pass.",
        "Leave-one-feature-out analysis shows that contract type and perceived security principally drive the profiles; sector formality adds theoretical context but little extra separation in this sample.",
        "The one routing-corrupted final-Branch-F record is retained in cleaned data but excluded from this analysis.",
    ],
}

with open(OUT / "employment_security_clustering_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved files:")
for path in sorted(OUT.iterdir()):
    print(" -", path.name)


{
  "cohort": {
    "records": 81,
    "branch": "F",
    "routing_source": "translated_final_active_branch",
    "strict_final_branch_records": 82,
    "excluded_routing_integrity_records": 1
  },
  "dimension": "employment security and formality",
  "clustering_features": [
    "contract_type",
    "job_security_12months",
    "sector_formality"
  ],
  "descriptors": [
    "employment_status",
    "contract_satisfaction",
    "hopes_to_change_job",
    "insecurity_bothers"
  ],
  "data_quality": {
    "records": 81,
    "unique_ids": 81,
    "wrong_branch": 0,
    "wrong_routing_source": 0,
    "path_disagreements_remaining": 0,
    "integrity_issues_remaining": 0,
    "invalid_activity_status": 0,
    "cluster_input_missing_values": 0,
    "invalid_security_codes": 0,
    "invalid_formality_codes": 0
  },
  "method": {
    "distance": "equal-weight mixed Gower on three original employment responses",
    "candidate_k": [
      2,
      3,
      4
    ],
    "selected_k": 2,
    "ens

## Interpretation guardrails

- These profiles describe contract arrangements, perceived job security, and
  sector formality.
- They do not measure personal worth, overall employability, programme
  suitability, or a validated quantity of job quality.
- Employment status, subjective contract satisfaction, intention to change
  jobs, and conditional security concern describe profiles but do not determine
  distance.
- A stable solution does not prove causal or permanent worker types.
- Profile names are descriptive summaries assigned after examining complete
  response distributions.
